In [1]:
import os
import re
import warnings
import gc
import torch
import pandas as pd
import numpy as np

# NLP & Visuals
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from nltk import pos_tag
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOP_WORDS

# BERTopic & Modeling
from bertopic import BERTopic
from bertopic.representation import BaseRepresentation
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Settings
warnings.filterwarnings('ignore')

# Download necessary NLTK data
nltk_dir = "../../../data/auto/"
os.makedirs(nltk_dir, exist_ok=True)
nltk.data.path.append(nltk_dir)
for pkg in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(pkg, download_dir=nltk_dir, quiet=True)

2026-03-22 14:46:52.577264: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# ==========================================
# Loading data
# ==========================================
try:
    df = pd.read_csv("../../../data/policy/combined_policy_docs_chunked.csv", encoding='utf-8-sig')
    df['Title'] = df['Title'].fillna('').astype(str)
    df = df[df['Content'] != 'N/A'].dropna(subset=['Content'])
    df['Content'] = df['Title'] + " " + df['Content'] 
    print(f"Successfully loaded {len(df)} documents.")
except FileNotFoundError:
    print("Error: File not found. Check path.")
    exit()

Successfully loaded 1172 documents.


In [3]:
# ==========================================
# Refined & consolidated stopwords
# ==========================================
stop_words = set(SPACY_STOP_WORDS).union(set(stopwords.words('english')))
try:
    stop_words.update(set(stopwords.words('french')))
except:
    pass

# ==========================================
# Consolidated refined stopwords for BERTopic & SLM
# ==========================================
pdf_structural_noise = [
    'pdf', 'qxp', 'page', 'copyright', 'reserved', 'www', 'http', 'https', 'com', 'org',
    'figure', 'table', 'annex', 'appendix', 'footnote', 'endnote', 'doi', 'isbn', 
    'bibliography', 'citation', 'reference', 'metadata', 'docx', 'executive summary', 
    'draft', 'matrix', 'blueprint', 'snapshot', 'chart', 'brochure', 'section', 'chapter', 
    'part', 'title', 'article', 'clause', 'paragraph', 'subparagraph', 'schedule', 
    'exhibit', 'supplement', 'addendum', 'preamble', 'recital', 'report', 'disposition',
    'alinea', 'expos', 'motifs', 'manual', 'handbook', 'procedure', 'form', 'file', 'save'
]

legal_vague_noise = [
    'shall', 'must', 'may', 'might', 'could', 'would', 'should', 'will', 'pursuant',
    'herein', 'therein', 'notwithstanding', 'accordance', 'furthermore', 'moreover',
    'hereby', 'whereby', 'hereinafter', 'thereafter', 'thereto', 'coppa', 'ferpa', 
    'enact', 'bit', 'actually', 'typical', 'various', 'several', 'final', 'total', 
    'sample', 'original', 'high', 'low', 'pro', 'stu', 'etc', 'august', 'april', 'july', 'session'
]

geo_admin_noise = [
    'ireland', 'irish', 'france', 'french', 'usa', 'america', 'australia', 'australian',
    'country', 'nation', 'national', 'state', 'government', 'federal', 'ministry', 'department',
    'gouvernement', 'ministre', 'dtat', 'republic', 'pittsburg', 'antioch', 'medina', 
    'winterset', 'sara', 'washington', 'california', 'massachusetts', 'oregon', 'sweden', 
    'unified', 'district', 'nsw', 'commonwealth', 'australasian', 'estonia', 'ny', 
    'new york', 'suffolk', 'long island', 'palo', 'alto', 'ca', 'ma', 'wa', 'northport', 
    'nenufsd', 'nenuf', 'sd', 'school district', 'administrative', 'administration', 
    'office', 'clerk', 'board', 'superintendent', 'enrollment', 'attendance'
]

domain_tech_noise = [
    'artificial', 'intelligence', 'ai', 'ia', 'genai', 'generative', 'model', 'models',
    'algorithm', 'algorithms', 'data', 'digital', 'technology', 'technological', 'software',
    'school', 'education', 'educational', 'pupil', 'classroom', 'learning', 'cole',
    'teaching', 'instruction', 'curriculum', 'student', 'teacher', 'educatif', 'enseignant',
    'professeur', 'programme', 'apprentissage', 'enseignement', 'formation', 'établissement',
    'covid', 'primary', 'technical', 'platform', 'online', 'internet', 'web', 'system', 
    'device', 'user', 'login', 'portal', 'dashboard', 'click', 'tab'
]

org_name_fragment_noise = [
    'adam', 'mary', 'jensen', 'abraham', 'lincoln', 'knight', 'kim', 'guez',
    'oecd', 'cbs', 'webwise', 'cosn', 'dlf', 'scoilnet', 'deap', 'dne', 'men', 
    'parliament', 'commissioner', 'public sector', 'cipa', 'protection act', 
    'oide', 'tie', 'cpd', 'pisa', 'ieee', 'openai', 'microsoft', 'google', 'amazon',
    'ing', 'tion', 'ment', 'ness', 'ly', 'able', 'ive', 'ent', 'ant', 'ence', 'ance', 
    'ali', 'gue', 'rod', 'rodr', 'educa', 'evi', 'showcase', 'deliverable', 'post', 'fig', 'fee'
]

all_custom_stops = (
    pdf_structural_noise + legal_vague_noise + 
    geo_admin_noise + domain_tech_noise + 
    org_name_fragment_noise
)

stop_words.update(all_custom_stops)

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'): return wordnet.ADJ
    elif treebank_tag.startswith('V'): return wordnet.VERB
    elif treebank_tag.startswith('N'): return wordnet.NOUN
    elif treebank_tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def clean_text(text):
    # Basic Validation
    if not isinstance(text, str) or len(text) < 5: return ""
    
    # French Apostrophe Handling (Alignment with TfidfVectorizer)
    # Replaces ' with a space (e.g., "l'éducation" -> "l éducation") 
    # so the vectorizer can recognize the core noun as a standalone token.
    text = re.sub(r"([a-zA-ZàâäéèêëïîôùûüÿçÀÂÄÉÈÊËÏÎÔÙÛÜŸÇ])'", r"\1 ", text)
    
    # Stop-Fragment Removal
    # Removes single-letter French articles/prepositions that lose meaning after splitting.
    text = re.sub(r"\b(l|d|c|j|qu)\b", "", text, flags=re.IGNORECASE)
    
    # Normalization & Special Character Stripping
    text = text.lower()
    # Matches the 'token_pattern' in your Vectorizer to keep accented characters 
    # essential for French policy terms (e.g., 'élèves', 'éthique').
    text = re.sub(r'[^a-zàâäéèêëïîôùûüÿç\s]', ' ', text)
    
    # Semantic Lemmatization
    # Uses POS tagging to reduce words to their dictionary root (e.g., 'teaching' -> 'teach').
    words = text.split()
    tagged = pos_tag(words)
    clean_words = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    
    # Final Filter
    # Removes stop words and ensures all tokens meet the minimum 3-character 
    # length requirement set in the TfidfVectorizer pattern.
    return " ".join([w for w in clean_words if w not in stop_words and len(w) > 2])

While **Latent Dirichlet Allocation (LDA)** and Satandard **BERTopic** pipelines are foundational tools in text mining, they often struggle with the linguistic nuances found in complex policy documents. Transitioning to a **BERTopic + SLM (Small Language Model)** framework addresses these limitations by moving from simple statistical counting to deep contextual understanding.

## Comparative advantages of BERTopic + SLM

### Beyond the "Bag-of-Words" (Advantage over LDA)

LDA operates on a "bag-of-words" principle, where the frequency of words determines the topic. This often leads to fragmented results in policy analysis where the same word (e.g., "support") might mean "technical IT support" in one document and "emotional student support" in another. LDA cannot distinguish between these contexts. BERTopic, using transformer-based embeddings, captures the **semantic meaning** of sentences, ensuring that topics are grouped by their actual intent rather than just shared vocabulary.

### Resolving keyword confusion (Advantage over pure BERTopic)

Pure BERTopic typically uses **TF-IDF** (Term Frequency-Inverse Document Frequency) to label topics. While mathematically sound, the resulting top 10 keywords can often be "confusing" or "noisy," especially when dealing with bilingual data (English/French) or methodological markers like **Interview** and **Excerpt**. 
* **BERTopic + TF-IDF:** Might present a topic as: `["education", "usage", "enseignants", "outils", "interview"]`.
* **BERTopic + SLM:** The SLM reads those specific keywords and synthesizes them into a coherent, human-readable label such as **"Practitioner Perspectives on Digital Tools."**

### Contextual synthesis and "Genre" recognition

The integration of an SLM (such as Qwen-0.5B) acts as an intelligent interpreter. It recognizes "genre signals", identifying when a cluster of words represents a formal mandate versus a qualitative interview. This allows the model to "clean" the output in real-time, discarding structural noise and highlighting the most significant policy themes. 

### Handling multi-lingual and technical nuance

In a global report involving different national "fingerprints," terminology varies significantly. Pure statistical models often create separate topics for the same concept if the words differ slightly. The SLM understands that `élèves` in a French document and `students` in a US document refer to the same entity, allowing for a unified global synthesis that LDA or standard TF-IDF might overlook.

In [4]:
# ==========================================
# Representation model (SLM) wrapper
# ==========================================
class LocalSLMWrapper(BaseRepresentation):
    """
    Custom wrapper to integrate a Local SLM (Small Language Model) 
    for automated topic labeling within BERTopic.
    """
    def __init__(self, pipe):
        self.pipeline = pipe

    def extract_topics(self, topic_model, documents, tf_idf, topics):
        topic_labels = {}
        for topic in topics.keys():
            # Handle noise/outliers (Topic -1)
            if topic == -1:
                topic_labels[topic] = ["Outliers"]
                continue
            
            # Extract top 10 keywords from c-TF-IDF to use as LLM prompt context
            words = [word for word, _ in topics[topic][:10]]
            keywords_str = ", ".join(words)
            prompt = f"Keywords: {keywords_str}\nPolicy Theme (3 words max):"
            
            try:
                # Generate concise labels using the SLM pipeline
                output = self.pipeline(prompt, max_new_tokens=15, do_sample=False)
                res = output[0]['generated_text'].split("Policy Theme (3 words max):")[-1].strip()
                clean_label = res.split('\n')[0].replace('"', '').strip()
                
                # Hallucination Guardrail: Reverts to top keywords if the LLM output 
                # contains filler phrases or lacks sufficient detail.
                # Expanded Hallucination Guardrail
                bad_patterns = [
                    # Conversational/Definitional Fillers
                    "type of", "is a", "refers to", "consists of", "related to", 
                    "associated with", "example of", "category of", "deals with",
                    
                    # Geographical/Administrative Noise (to avoid hyper-local labeling)
                    "located", "state of", "district", "county", "region of", 
                    "territory", "city of", "department of",
                    
                    # Meta-talk (the LLM describing its own output)
                    "policy theme", "keywords include", "the theme is", "summary:", 
                    "topic involves", "here is", "this is",
                    
                    # Overly Broad/Vague Terms
                    "various", "multiple", "etc", "and so on", "general", "aspects",
                    
                    # Structural/Methodological Noise
                    "interview with", "excerpt from", "document", "source", "page", 
                    "table", "section", "paragraph"
                ]
                if any(p in clean_label.lower() for p in bad_patterns) or len(clean_label) < 3:
                    clean_label = f"{words[0].title()} & {words[1].title()}"
                
                print(f"Topic {topic} Keywords: {keywords_str[:50]}... -> Label: {clean_label}")
                topic_labels[topic] = [clean_label]
            except:
                # Fallback mechanism in case of generation failure
                topic_labels[topic] = [f"{words[0]} {words[1]}"]
                
        return topic_labels

In [5]:
# ==========================================
# Initialize models & components
# ==========================================

# Hardware acceleration check
# Detects if a GPU (CUDA) is available to significantly speed up embedding and 
# generation; defaults to CPU if no dedicated graphics hardware is present.
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Multilingual semantic embedding model
# Loads a transformer model optimized for mapping sentences into a shared vector 
# space. The 'multilingual-MiniLM' is chosen for its balance of speed and 
# accuracy in handling cross-lingual policy themes (e.g., English and French).
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Small Language Model (SLM) configuration
# Utilizes Qwen2-0.5B-Instruct, a compact but powerful model designed to 
# interpret clusters of text and generate human-readable labels.
model_id = "Qwen/Qwen2-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Model loading with memory optimization
# Loads the causal language model with float16 precision (on GPU) to reduce 
# VRAM usage while maintaining the thematic nuance required for policy labeling.
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.float16 if device=="cuda" else torch.float32
).to(device)

# 5. Pipeline & custom wrapper integration
# Wraps the SLM in a text-generation pipeline and connects it to the 
# LocalSLMWrapper, allowing BERTopic to use 'intelligent' synthesis for 
# topic discovery rather than relying solely on raw keyword lists.
slm_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0 if device=="cuda" else -1)
slm_labeler = LocalSLMWrapper(slm_pipe)

Device set to use cpu


In [6]:
# ==========================================
# Global topic discovery
# ==========================================

print("\nStarting global discovery")
# Prepare documents for processing
raw_docs = df['Content'].tolist()
# Clean text to remove HTML/structural noise while keeping 'genre signals'
cleaned_docs = [clean_text(doc) for doc in raw_docs]

# 1. Global Embeddings
# Convert text into 384-dimensional vectors using the multilingual transformer.
# This ensures that semantically similar concepts (e.g., 'teacher' vs 'enseignant') 
# are mathematically close in the vector space.
embeddings = embedding_model.encode(cleaned_docs, show_progress_bar=True)

# 2. Dimensionality Reduction (UMAP)
# Condenses high-dimensional vectors into 5 dimensions for efficient clustering.
# Uses 'cosine' metric to focus on semantic direction rather than document length.
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# 3. Clustering (HDBSCAN)
# Automatically identifies dense clusters of documents. 
# min_cluster_size=25 ensures that discovered topics have significant evidence.
hdbscan_model = HDBSCAN(min_cluster_size=25, min_samples=5, metric='euclidean', cluster_selection_method='eom')

# 4. Text Vectorization (TF-IDF)
# This component defines the linguistic boundaries for topic extraction. 
# By setting ngram_range=(1, 2), the model recognizes multi-word concepts—
# such as 'digital literacy' or 'cadre éthique'—as single semantic units. 
# The specific regex pattern acts as a final quality filter, ensuring only 
# substantive terms of 3+ characters (including essential French accents) 
# are passed to the SLM for thematic labeling.
vectorizer_model = TfidfVectorizer(
    ngram_range=(1, 2), 
    stop_words=list(stop_words), 
    # Use a simpler pattern since clean_text already handled the logic
    token_pattern=r"(?u)\b\w{3,}\b" 
)

# 5. The BERTopic Pipeline
# Integrates all components, crucially using 'llm_labeler' (the SLM) to 
# transform statistical word lists into professional policy themes.
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=slm_labeler,
    verbose=True
)

# Fit the model and extract global topics
topics, _ = topic_model.fit_transform(raw_docs, embeddings)

# Retrieve the final results with SLM-synthesized labels
topic_info = topic_model.get_topic_info()


Starting global discovery


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

2026-03-22 14:51:34,564 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-22 14:51:43,448 - BERTopic - Dimensionality - Completed ✓
2026-03-22 14:51:43,449 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-22 14:51:43,480 - BERTopic - Cluster - Completed ✓
2026-03-22 14:51:43,483 - BERTopic - Representation - Fine-tuning topics using representation models.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Topic 0 Keywords: students, programming, cyberbullying, games, resul... -> Label: Cyberbullying
Topic 1 Keywords: cluster, sef, project, clusters, evaluation 2025, ... -> Label: Evaluation
Topic 2 Keywords: élèves, outils, usages, éducation, enseignants, sc... -> Label: Education through technology
Topic 3 Keywords: interview excerpt, interview, excerpt, teachers, u... -> Label: teacher feedback, student learning, classroom management, technology integration
Topic 4 Keywords: use, policy, staff, tools, students, guidance, pri... -> Label: Privacy Policy
Topic 5 Keywords: students, teachers, ai_report_us_gov, constituents... -> Label: AI in education
Topic 6 Keywords: schools, strategy schools, strategy, 2027, schools... -> Label: Strategy
Topic 7 Keywords: teachers, competency, cft, tools, human, unesco_ai... -> Label: Competency Framework for Teachers
Topic 8 Keywords: principles, principles policy, policy, law, policy... -> Label: Principles Policy Practice
Topic 9 Keywords: offloadi

2026-03-22 14:53:33,772 - BERTopic - Representation - Completed ✓


Topic 14 Keywords: dna, 1202, literacy, 2202, yim, dai, 2021, shamir,... -> Label: DNA


In [7]:
# ==========================================
# Saving outputs
# ==========================================

# 1. Output Directory Management
# Establishes a dedicated path for policy data exports and ensures the 
# directory structure exists before attempting to write files.
OUTPUT_DIR = "../../../data/policy/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. SLM Label Extraction
# BERTopic stores multiple representations (c-TF-IDF, LLM, etc.). This 
# logic specifically extracts the primary 'intelligent' label generated 
# by the SLM from the 'Representation' list to use as the official title.
topic_info = topic_model.get_topic_info()
if "Representation" in topic_info.columns:
    topic_info['Topic_Label'] = topic_info['Representation'].apply(lambda x: x[0] if isinstance(x, list) else x)

# 3. Document-Level Mapping
# Assigns each original document its specific Topic ID and the 
# human-readable label generated by the SLM, bridging the gap between 
# raw text and thematic categorization.
df['Topic_ID'] = topics
df['Topic_Label'] = df['Topic_ID'].map(dict(zip(topic_info['Topic'], topic_info['Topic_Label'])))

# 4. Final Export
# Saves the full dataset (for per-country filtering later) and the global 
# summary. 'utf-8-sig' is used to ensure French accents (é, à, ç) render 
# correctly in Excel and other reporting tools.
df.to_csv(os.path.join(OUTPUT_DIR, "global_assignments.csv"), index=False, encoding='utf-8-sig')
topic_info.to_csv(os.path.join(OUTPUT_DIR, "global_summary.csv"), index=False, encoding='utf-8-sig')

print("\nPipeline complete. Files saved to ../../../data/policy/")


Pipeline complete. Files saved to ../../../data/policy/


## Global BERTopic + SLM Analysis

The BERTopic + SLM pipeline has generated a **high-resolution map of 15 distinct topics**, offering a level of granularity that the 4-topic LDA model could not achieve. The use of a SLM for labeling has successfully synthesized semantic clusters into descriptive, human-readable titles, cutting through the "keyword soup" typical of standard models.

### Key interpretations of the 15 Topics

*   **Specialized risk & safety (Topics 0 & 13):** The model isolated **"Cyberbullying"** (Topic 0) and **"Disability"** (Topic 13) as standalone themes. This indicates that safety protocols, student wellbeing, and inclusion (specifically regarding disabled people) are distinct, voluminous conversations within the corpus, rather than being sub-components of general policy.
*   **Operational specificity (Topic 1):** The **"Evaluation"** topic (containing `cluster`, `sef`, `project`) accurately captures the Irish-specific administrative language seen in the earlier country analysis, which LDA often conflated with general research.
*   **Genre recognition (Topic 3):** A major success of the SLM labeling is Topic 3: **"teacher feedback, student learning, classroom management, technology integration."** The SLM correctly identified that the underlying keywords (`interview excerpt`, `teachers`) represent *qualitative feedback* and *practitioner voice*, separating this "genre" from formal policy documents.
*   **Nuanced cognitive science (Topic 9):** Instead of a broad "Cognitive Research" label, the SLM identified **"Offloading."** This demonstrates a deep semantic understanding of the text, distinguishing the specific pedagogical theory of *cognitive offloading* from general educational psychology.
*   **Framework & governance separation:** The global LDA grouped privacy, ethics, and strategy into one "Governance" bucket. BERTopic + SLM splits this into **"Privacy Policy"** (Topic 4), **"Strategy"** (Topic 6), and **"Principles Policy Practice"** (Topic 8), allowing for a precise analysis of compliance versus high-level planning.
*   **Noise isolation (Topic 14):** The model identified **"DNA"** (Topic 14) as a distinct topic. Looking at the keywords (`yim`, `dai`, `shamir`), this is clearly noise/artifact text (likely reversed words from a PDF scrape). The model correctly quarantined this garbage, preventing it from polluting legitimate topics.

### Comparative analysis: BERTopic + SLM vs. Global LDA

The transition from Global LDA (4 topics) to BERTopic + SLM (15 topics) represents a shift from a **satellite view** to a **street-level view** of the policy landscape.

#### Granularity vs. Generalization
*   **LDA:** Produced broad pillars like **"Governance & Ethics"**. While useful for a high-level summary, this obscured specific policy instruments. For instance, LDA buried the specific issue of "cyberbullying" inside "Institutional Evaluation" or "Research," making it invisible.
*   **BERTopic + SLM:** Successfully extracted **"Cyberbullying"** and **"Privacy Policy"** as separate entities. This allows analysts to query specific safety concerns rather than wading through a general "ethics" topic.

#### Semantic precision vs. "Bag of Words"
*   **LDA:** Relied on word co-occurrence. It created a **"Cognitive Research"** topic that mixed general literacy with specific psychological theories. It could not distinguish between a document about *reading skills* and a document about *cognitive load theory*.
*   **BERTopic + SLM:** Used embeddings to understand context. It separated **"Education through technology"** (general pedagogy) from **"Offloading"** (specific cognitive theory). This distinction is critical for understanding the *depth* of discourse in the documents.

#### Handling multilingualism
*   **LDA:** The global LDA output had a bilingual topic (**"Pedagogical Practice"**) but labeled it descriptively only in the analysis, not the model output.
*   **BERTopic + SLM:** The SLM successfully synthesized French keywords (`élèves`, `outils`) into an English label: **"Education through technology."** It bridged the language gap effectively, treating French and English inputs on equal semantic footing.

#### Genre identification (The "Interview" Factor)
*   **LDA:** Often mixed *methodology* with *theme*. Because the word "interview" appeared frequently, LDA associated it with "Institutional Evaluation," creating a confusing topic that suggested evaluation *is* interviewing.
*   **BERTopic + SLM:** The SLM recognized the *genre* of the text. It labeled Topic 3 based on the *nature* of the content (**"teacher feedback... classroom management"**) rather than just the keywords. It correctly identified that these are qualitative excerpts, distinct from regulatory policy.

### Conclusion
The BERTopic + SLM pipeline validates the limitations of LDA in this context. While LDA provided a valid "4-pillar" structure of the global landscape, it failed to capture the operational nuances (like evaluation clusters), the specific risks (like cyberbullying), and the distinct document genres (like interview feedback). BERTopic + SLM bridges this gap by providing a **multi-faceted taxonomy** that separates *compliance* (Privacy) from *strategy* (Strategic planning) and *safety* (Cyberbullying) from *theory* (Offloading), offering a much richer foundation for decision-making.